# Metadata-only baseline and job-title ablation (camera-ready CR-5, CR-6)

Answers three reviewer requests in one place:
- **CR-6 (R2):** a *metadata-only* baseline, so text-only vs metadata-only vs
  text+metadata can be compared directly. SHAP shows metadata dominates the best
  model; this tells us how much the text adds on top.
- **CR-5 (R1, R3):** `Speaker's job title` is a near-unique speaker proxy
  ("President-Elect" is one person). Two ablations drop it (and then also `Context`,
  the other near-unique column) to see whether the metadata gain survives without it.
- **CR-5, speaker-disjoint:** every feature space is also scored under
  speaker-disjoint 5-fold CV against a matched random 5-fold control on the same
  pooled rows. The *gap* (random minus disjoint) measures how much a score depends on
  having seen the same speakers in training.

**Official-split numbers** use exactly the Table III procedure
(`proposed_with_metadata_v1.ipynb`): one Pipeline (metadata canonicalizer -> TF-IDF +
metadata encoders -> classifier) tuned by `GridSearchCV` (5-fold, macro-F1, same
grids), so every feature step is refit inside each CV fold (CR-7); the final model is
refit on the 11,524-row train+valid pool and the test set is scored once. The NB
text+metadata row must therefore equal Table III's; this is asserted below.

**Speaker-disjoint numbers** refit all features inside every fold and reuse the
hyperparameters selected on the official pool, so the two protocols differ only in how
rows are grouped into folds.

No p-values are computed here. Test-set predictions are saved for the joint,
corrected significance analysis (CR-4).

In [1]:
import re
import warnings

import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from scipy.sparse import hstack
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold, StratifiedKFold
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

from liar_utils import RANDOM_STATE, evaluate_full, load_and_label
from metadata_features import METADATA_COLUMNS, MetadataCanonicalizer, build_metadata_features, build_pipeline_transformer

warnings.filterwarnings("ignore", category=UserWarning)

## Data

In [2]:
train_raw = load_and_label("train.csv")
valid_raw = load_and_label("valid.csv")
test = load_and_label("test.csv")
train = pd.concat([train_raw, valid_raw], ignore_index=True)  # same 11,524-row pool as Table III
assert (len(train), len(test)) == (11524, 1267)

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()


def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z]", " ", text)
    return " ".join(stemmer.stem(w) for w in text.split() if w not in stop_words)


train["clean_text"] = train["Statement"].apply(preprocess)
test["clean_text"] = test["Statement"].apply(preprocess)
y_train, y_test = train["Label"].to_numpy(), test["Label"].to_numpy()

NO_JOB = [c for c in METADATA_COLUMNS if c != "Speaker's job title"]
NO_JOB_CONTEXT = [c for c in NO_JOB if c != "Context"]
# name -> (use text?, metadata columns or None)
FEATURE_SPACES = {
    "text": (True, None),
    "metadata": (False, METADATA_COLUMNS),
    "text+metadata": (True, METADATA_COLUMNS),
    "text+metadata -job": (True, NO_JOB),
    "text+metadata -job -context": (True, NO_JOB_CONTEXT),
}
MODELS = {
    "Naive Bayes": (MultinomialNB(), {"alpha": [0.1, 0.5, 1.0]}),
    "Logistic Regression": (
        LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        {"C": [0.1, 1, 10], "solver": ["liblinear"], "class_weight": [None, "balanced"]},
    ),
}

## How strongly does job title identify the speaker? (CR-5)

In [3]:
job = train["Speaker's job title"].fillna("unknown").astype(str).str.strip()
known = job != "unknown"
speakers_per_title = train[known].groupby(job[known])["Speaker"].nunique()
single = speakers_per_title[speakers_per_title == 1].index
job_stats = {
    "distinct job titles (train+valid)": len(speakers_per_title),
    "titles held by exactly one speaker": int((speakers_per_title == 1).sum()),
    "share of titles held by one speaker": (speakers_per_title == 1).mean(),
    "rows whose title is held by one speaker": int(job[known].isin(single).sum()),
    "share of all rows": job[known].isin(single).sum() / len(train),
    "rows with missing title": int((~known).sum()),
}
for t in ["President-Elect", "President"]:
    counts = train.loc[job == t, "Speaker"].value_counts()
    job_stats[f"'{t}' rows (speakers)"] = f"{counts.sum()} ({dict(counts.head(3))})"
job_stats_table = pd.Series(job_stats, name="value").to_frame()
job_stats_table.to_csv("jobtitle_speaker_stats_v1.csv")
job_stats_table

,value
distinct job titles (train+valid),1258
titles held by exactly one speaker,1136
share of titles held by one speaker,0.903021
rows whose title is held by one speaker,3595
share of all rows,0.311958
rows with missing title,3243
'President-Elect' rows (speakers),310 ({'donald-trump': np.int64(310)})
'President' rows (speakers),"553 ({'barack-obama': np.int64(549), 'george-w..."


## Feature builder
Fits TF-IDF and the chosen metadata columns on `fit_df`, transforms both frames.
Same TF-IDF settings and metadata encoders as Table III.

In [4]:
def build_features(fit_df, apply_df, use_text, meta_cols):
    blocks_fit, blocks_apply = [], []
    if use_text:
        tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2, max_df=0.9)
        blocks_fit.append(tfidf.fit_transform(fit_df["clean_text"]))
        blocks_apply.append(tfidf.transform(apply_df["clean_text"]))
    if meta_cols is not None:
        m_fit, m_apply, _ = build_metadata_features(fit_df, apply_df, include_speaker=False, columns=meta_cols)
        blocks_fit.append(m_fit)
        blocks_apply.append(m_apply)
    return hstack(blocks_fit).tocsr(), hstack(blocks_apply).tocsr()


def make_pipeline(use_text, meta_cols, clf):
    steps = []
    if meta_cols is not None:
        steps.append(("canon", MetadataCanonicalizer(columns=meta_cols)))
    steps.append(("features", build_pipeline_transformer(
        use_metadata=meta_cols is not None, columns=meta_cols or METADATA_COLUMNS, use_text=use_text)))
    steps.append(("clf", clf))
    return Pipeline(steps)


def bootstrap_ci(y_true, y_pred, n_boot=2000, seed=RANDOM_STATE):
    rng = np.random.RandomState(seed)
    n = len(y_true)
    scores = [f1_score(y_true[idx], y_pred[idx], average="macro") for idx in (rng.randint(0, n, n) for _ in range(n_boot))]
    return np.percentile(scores, [2.5, 97.5])

## Official split: fit on the 11,524-row pool, score the 1,267-row test set once

In [5]:
official_rows, best_params = [], {}
predictions = pd.DataFrame({"ID": test["ID"].values, "y_true": y_test})
for space, (use_text, meta_cols) in FEATURE_SPACES.items():
    for model_name, (model, grid_params) in MODELS.items():
        pipe = make_pipeline(use_text, meta_cols, clone(model))
        grid = GridSearchCV(pipe, {f"clf__{k}": v for k, v in grid_params.items()}, cv=5, scoring="f1_macro", n_jobs=-1)
        grid.fit(train, y_train)
        y_pred = grid.best_estimator_.predict(test)
        n_features = grid.best_estimator_[:-1].transform(test.head(1)).shape[1]
        m = evaluate_full(y_test, y_pred)
        lo, hi = bootstrap_ci(y_test, y_pred)
        best_params[(model_name, space)] = {k.removeprefix("clf__"): v for k, v in grid.best_params_.items()}
        predictions[f"pred__{model_name}__{space}"] = y_pred
        official_rows.append({
            "Model": model_name, "Feature space": space, "Features": n_features,
            "CV Macro-F1 Mean": grid.cv_results_["mean_test_score"][grid.best_index_],
            "CV Macro-F1 Std": grid.cv_results_["std_test_score"][grid.best_index_],
            "Test Accuracy": m["accuracy"], "Test Macro-F1": m["macro_f1"],
            "Test Macro-F1 CI low": lo, "Test Macro-F1 CI high": hi,
            "Test Fake F1": m["fake_f1"], "Test Real F1": m["real_f1"], "Best Params": best_params[(model_name, space)],
        })
        print(f"{model_name:20s} {space:28s} test macro-F1 {m['macro_f1']:.4f}  [{lo:.3f}, {hi:.3f}]  {best_params[(model_name, space)]}")

official = pd.DataFrame(official_rows)
nb_tm = official.query("Model == 'Naive Bayes' and `Feature space` == 'text+metadata'")["Test Macro-F1"].item()
table3 = pd.read_csv("model_comparison_results_v3_metadata.csv").query("Model == 'Naive Bayes' and Method == 'Text + metadata'")["Test Macro-F1"].item()
assert abs(nb_tm - table3) < 1e-9, f"NB text+metadata ({nb_tm:.4f}) must equal Table III's ({table3:.4f})"
official.to_csv("metadata_ablation_results_v1.csv", index=False)
predictions.to_csv("metadata_ablation_test_predictions_v1.csv", index=False)
official.pivot(index="Feature space", columns="Model", values="Test Macro-F1").loc[list(FEATURE_SPACES)].round(4)

Naive Bayes          text                         test macro-F1 0.6035  [0.575, 0.631]  {'alpha': 0.5}


Logistic Regression  text                         test macro-F1 0.6153  [0.587, 0.642]  {'C': 0.1, 'class_weight': 'balanced', 'solver': 'liblinear'}


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parame

Naive Bayes          metadata                     test macro-F1 0.6176  [0.592, 0.644]  {'alpha': 1.0}


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parame

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parame

... (21 outputs left out)

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parame

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parame

Logistic Regression  text+metadata -job -context  test macro-F1 0.6265  [0.599, 0.653]  {'C': 1, 'class_weight': 'balanced', 'solver': 'liblinear'}


Model,Logistic Regression,Naive Bayes
Feature space,,
text,0.6153,0.6035
metadata,0.6193,0.6176
text+metadata,0.6229,0.6549
text+metadata -job,0.6190,0.6316
text+metadata -job -context,0.6265,0.6321


## Speaker-disjoint vs matched random 5-fold CV on all 12,791 pooled rows
Hyperparameters are fixed to the official-pool choice above. Features are refit
inside every fold.

In [6]:
pooled = pd.concat([train, test], ignore_index=True)
y_pool = pooled["Label"].to_numpy()
groups = pooled["Speaker"].fillna("unknown").astype(str).str.strip().to_numpy()
protocols = {
    "random": list(StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE).split(pooled, y_pool)),
    "speaker-disjoint": list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE).split(pooled, y_pool, groups)),
}
for tr_idx, te_idx in protocols["speaker-disjoint"]:
    assert not set(groups[tr_idx]) & set(groups[te_idx]), "speaker overlap in a speaker-disjoint fold"
overlap_random = np.mean([np.isin(groups[te], groups[tr]).mean() for tr, te in protocols["random"]])
print(f"Random folds: {overlap_random:.1%} of held-out rows share a speaker with training; speaker-disjoint: 0%")

fold_rows = []
for space, (use_text, meta_cols) in FEATURE_SPACES.items():
    for model_name, (model, _) in MODELS.items():
        for protocol, folds in protocols.items():
            for k, (tr_idx, te_idx) in enumerate(folds):
                fit_df, held_df = pooled.iloc[tr_idx], pooled.iloc[te_idx]
                X_tr, X_te = build_features(fit_df, held_df, use_text, meta_cols)
                clf = clone(model).set_params(**best_params[(model_name, space)]).fit(X_tr, y_pool[tr_idx])
                fold_rows.append({"Model": model_name, "Feature space": space, "Protocol": protocol, "Fold": k,
                                  "Macro-F1": f1_score(y_pool[te_idx], clf.predict(X_te), average="macro")})
        print(f"done: {space}")

folds_df = pd.DataFrame(fold_rows)
folds_df.to_csv("speaker_disjoint_folds_v1.csv", index=False)
summary = folds_df.groupby(["Model", "Feature space", "Protocol"])["Macro-F1"].agg(["mean", "std"]).unstack("Protocol")
summary.columns = [f"{proto} {stat}" for stat, proto in summary.columns]
summary["gap (random - disjoint)"] = summary["random mean"] - summary["speaker-disjoint mean"]
summary = summary.reindex(pd.MultiIndex.from_product([list(MODELS), list(FEATURE_SPACES)], names=["Model", "Feature space"]))
summary.to_csv("speaker_disjoint_summary_v1.csv")
summary.round(4)

Random folds: 81.9% of held-out rows share a speaker with training; speaker-disjoint: 0%


done: text


done: text


done: metadata


done: metadata


done: text+metadata


... (2 outputs left out)

done: text+metadata -job


done: text+metadata -job -context


done: text+metadata -job -context


random mean  \
Model               Feature space                              
Naive Bayes         text                              0.5938   
                    metadata                          0.5973   
                    text+metadata                     0.6271   
                    text+metadata -job                0.6174   
                    text+metadata -job -context       0.6173   
Logistic Regression text                              0.6046   
                    metadata                          0.5970   
                    text+metadata                     0.6222   
                    text+metadata -job                0.6206   
                    text+metadata -job -context       0.6232   

                                                 speaker-disjoint mean  \
Model               Feature space                                        
Naive Bayes         text                                        0.5817   
                    metadata                                    0.5675   
                    text+metadata                               0.6016   
                    text+metadata -job                          0.5975   
                    text+metadata -job -context                 0.5971   
Logistic Regression text                                        0.6020   
                    metadata                                    0.5806   
                    text+metadata                               0.6062   
                    text+metadata -job                          0.5992   
                    text+metadata -job -context                 0.6036   

                                                 random std  \
Model               Feature space                             
Naive Bayes         text                             0.0148   
                    metadata                         0.0071   
                    text+metadata                    0.0083   
                    text+metadata -job               0.0102   
                    text+metadata -job -context      0.0130   
Logistic Regression text                             0.0090   
                    metadata                         0.0063   
                    text+metadata                    0.0149   
                    text+metadata -job               0.0142   
                    text+metadata -job -context      0.0150   

                                                 speaker-disjoint std  \
Model               Feature space                                       
Naive Bayes         text                                       0.0139   
                    metadata                                   0.0084   
                    text+metadata                              0.0038   
                    text+metadata -job                         0.0076   
                    text+metadata -job -context                0.0053   
Logistic Regression text                                       0.0117   
                    metadata                                   0.0050   
                    text+metadata                              0.0163   
                    text+metadata -job                         0.0101   
                    text+metadata -job -context                0.0092   

                                                 gap (random - disjoint)  
Model               Feature space                                         
Naive Bayes         text                                          0.0121  
                    metadata                                      0.0298  
                    text+metadata                                 0.0255  
                    text+metadata -job                            0.0199  
                    text+metadata -job -context                   0.0202  
Logistic Regression text                                          0.0026  
                    metadata                                      0.0164  
                    text+metadata                                 0.0160  
       

## Readout: gains over text-only, under both protocols

In [7]:
readout = []
for model_name in MODELS:
    off = official[official["Model"] == model_name].set_index("Feature space")["Test Macro-F1"]
    dis = summary.loc[model_name, "speaker-disjoint mean"]
    for space in FEATURE_SPACES:
        readout.append({"Model": model_name, "Feature space": space,
                        "Official test": off[space], "vs text (official)": off[space] - off["text"],
                        "Speaker-disjoint CV": dis[space], "vs text (disjoint)": dis[space] - dis["text"]})
readout = pd.DataFrame(readout)
readout.to_csv("metadata_ablation_readout_v1.csv", index=False)
readout.round(4)

,Model,Feature space,Official test,vs text (official),Speaker-disjoint CV,vs text (disjoint)
0,Naive Bayes,text,0.6035,0.0000,0.5817,0.0000
1,Naive Bayes,metadata,0.6176,0.0140,0.5675,-0.0142
2,Naive Bayes,text+metadata,0.6549,0.0514,0.6016,0.0199
3,Naive Bayes,text+metadata -job,0.6316,0.0281,0.5975,0.0158
4,Naive Bayes,text+metadata -job -context,0.6321,0.0286,0.5971,0.0153
5,Logistic Regression,text,0.6153,0.0000,0.6020,0.0000
6,Logistic Regression,metadata,0.6193,0.0039,0.5806,-0.0214
7,Logistic Regression,text+metadata,0.6229,0.0076,0.6062,0.0042
8,Logistic Regression,text+metadata -job,0.6190,0.0037,0.5992,-0.0028
9,Logistic Regression,text+metadata -job -context,0.6265,0.0111,0.6036,0.0016
